23102A0007- Aarti Sakpal

Lab 3: Multi-Source Retail Sales Data Integration and Analysis
Objective

To import, clean, integrate, analyze, and store heterogeneous retail sales data from CSV, JSON, and Excel sources using R and SQLite.

Dataset

UCI Online Retail Dataset

The original dataset contains transaction-level retail sales information.

Data Sources Created
transactions.csv — InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate
products.json — StockCode, Description, UnitPrice
customers.xlsx — CustomerID, Country

In [2]:
install.packages(c(
  "tidyverse",
  "jsonlite",
  "readxl",
  "writexl",
  "DBI",
  "RSQLite"
))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [3]:
library(tidyverse)
library(jsonlite)
library(readxl)
library(writexl)
library(DBI)
library(RSQLite)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘jsonlite’


The following object is masked from ‘package:purrr’:

    flatten




Task 1: Data Import and Cleaning

The UCI Online Retail dataset was imported into R using read_excel().

The dataset was inspected for:

Missing values
Duplicate records
Invalid or zero quantities
Invalid or zero unit prices
Cleaning Decisions
Records with missing CustomerID were removed because they could not be reliably associated with a customer.
Records with missing Description were removed because product identification was incomplete.
Duplicate records were removed using distinct().
Records with Quantity <= 0 were removed because they do not represent valid sales transactions.
Records with UnitPrice <= 0 were removed because they cannot contribute to valid sales revenue.
Revenue was calculated as:

Revenue = Quantity × UnitPrice

In [4]:
download.file(
  "https://archive.ics.uci.edu/static/public/352/online+retail.zip",
  "online_retail.zip"
)

unzip("online_retail.zip")

In [5]:
list.files()

[1] "Online Retail.xlsx" "online_retail.zip"  "sample_data"

In [6]:
retail_raw <- read_excel("Online Retail.xlsx")

In [7]:
head(retail_raw)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


In [8]:
dim(retail_raw)

[1] 541909      8

In [9]:
str(retail_raw)

tibble [541,909 × 8] (S3: tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr [1:541909] "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 $ UnitPrice  : num [1:541909] 2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Country    : chr [1:541909] "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


In [10]:
summary(retail_raw)

     InvoiceNo          StockCode         Description        Quantity         
 Length   :541909   Length   :541909   Length   :541909   Min.   :-80995.000  
 N.unique : 25900   N.unique :  4070   N.unique :  4211   1st Qu.:     1.000  
 N.blank  :     0   N.blank  :     0   N.blank  :     0   Median :     3.000  
 Min.nchar:     6   Min.nchar:     1   Min.nchar:     1   Mean   :     9.552  
 Max.nchar:     7   Max.nchar:    12   Max.nchar:    35   3rd Qu.:    10.000  
                                       NAs      :  1454   Max.   : 80995.000  
                                                                              
  InvoiceDate                    UnitPrice            CustomerID    
 Min.   :2010-12-01 08:26:00   Min.   :-11062.060   Min.   :12346   
 1st Qu.:2011-03-28 11:34:00   1st Qu.:     1.250   1st Qu.:13953   
 Median :2011-07-19 17:17:00   Median :     2.080   Median :15152   
 Mean   :2011-07-04 13:34:57   Mean   :     4.611   Mean   :15288   
 3rd Qu.:2011-10-19 11:

In [11]:
colSums(is.na(retail_raw))

InvoiceNo   StockCode Description    Quantity InvoiceDate   UnitPrice 
          0           0        1454           0           0           0 
 CustomerID     Country 
     135080           0

In [12]:
retail_clean <- retail_raw %>%
  filter(
    !is.na(Description),
    !is.na(CustomerID)
  )

In [13]:
dim(retail_clean)

[1] 406829      8

In [14]:
sum(duplicated(retail_clean))

[1] 5225

In [15]:
retail_clean <- retail_clean %>%
  distinct()

In [16]:
sum(duplicated(retail_clean))

[1] 0

In [17]:
dim(retail_clean)

[1] 401604      8

In [18]:
sum(retail_clean$Quantity <= 0)

[1] 8872

In [19]:
sum(retail_clean$Quantity == 0)

[1] 0

In [20]:
sum(retail_clean$Quantity < 0)

[1] 8872

In [21]:
retail_clean <- retail_clean %>%
  filter(Quantity > 0)

In [22]:
sum(retail_clean$Quantity <= 0)

[1] 0

In [23]:
dim(retail_clean)

[1] 392732      8

In [24]:
sum(retail_clean$UnitPrice <= 0)

[1] 40

In [25]:
sum(retail_clean$UnitPrice == 0)

[1] 40

In [26]:
retail_clean <- retail_clean %>%
  filter(UnitPrice > 0)

In [27]:
sum(retail_clean$UnitPrice <= 0)

[1] 0

In [28]:
dim(retail_clean)

[1] 392692      8

In [29]:
retail_clean <- retail_clean %>%
  filter(UnitPrice > 0)

In [30]:
sum(retail_clean$UnitPrice <= 0)

[1] 0

In [31]:
dim(retail_clean)

[1] 392692      8

Cleaning Results
Stage	Records
Original dataset	541,909

After missing-value removal	406,829

After duplicate removal	401,604

After invalid quantity removal	392,732

After invalid price removal	392,692

Final Cleaning Verification

Missing Description = 0

Missing CustomerID = 0

Duplicate records = 0

Invalid Quantity = 0

Invalid UnitPrice = 0

Final records = 392,692


In [32]:
list(
  Missing_Description = sum(is.na(retail_clean$Description)),
  Missing_CustomerID = sum(is.na(retail_clean$CustomerID)),
  Duplicate_Rows = sum(duplicated(retail_clean)),
  Invalid_Quantity = sum(retail_clean$Quantity <= 0),
  Invalid_UnitPrice = sum(retail_clean$UnitPrice <= 0),
  Final_Rows = nrow(retail_clean)
)

$Missing_Description
[1] 0

$Missing_CustomerID
[1] 0

$Duplicate_Rows
[1] 0

$Invalid_Quantity
[1] 0

$Invalid_UnitPrice
[1] 0

$Final_Rows
[1] 392692

In [33]:
transactions <- retail_clean %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate
  )

In [34]:
head(transactions)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


In [35]:
dim(transactions)

[1] 392692      5

In [36]:
write_csv(transactions, "transactions.csv")

In [37]:
list.files()

[1] "Online Retail.xlsx" "online_retail.zip"  "sample_data"       
[4] "transactions.csv"

In [38]:
products <- retail_clean %>%
  select(
    StockCode,
    Description,
    UnitPrice
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

In [40]:
head(products)

StockCode,Description,UnitPrice
<chr>,<chr>,<dbl>
85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
71053,WHITE METAL LANTERN,3.39
84406B,CREAM CUPID HEARTS COAT HANGER,2.75
84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
22752,SET 7 BABUSHKA NESTING BOXES,7.65


In [41]:
dim(products)

[1] 3665    3

In [42]:
write_json(
  products,
  "products.json",
  pretty = TRUE,
  auto_unbox = TRUE
)

In [43]:
list.files()

[1] "Online Retail.xlsx" "online_retail.zip"  "products.json"     
[4] "sample_data"        "transactions.csv"

Task 2: Data Integration

The three data sources were integrated using left_join().

Join Keys
Transactions + Products → StockCode
Transactions + Customers → CustomerID
Join Justification

left_join() was selected because the transaction dataset is the primary dataset. It ensures that all valid transaction records are retained while matching product and customer information is added where available.

Integration Results

Transactions	392,692	5

Products	3,665	3

Customers	4,338	2

Final integrated dataset	392,692	10

Unmatched Records

Unmatched products = 0

Unmatched customers = 0


The final integrated dataset contains:

InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate, Description, UnitPrice, Revenue, Country, Customer_Category

In [44]:
customers <- retail_clean %>%
  select(
    CustomerID,
    Country
  ) %>%
  distinct(CustomerID, .keep_all = TRUE)

In [45]:
head(customers)

CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


In [46]:
dim(customers)

[1] 4338    2

In [47]:
write_xlsx(
  customers,
  "customers.xlsx"
)

In [48]:
list.files()

[1] "customers.xlsx"     "Online Retail.xlsx" "online_retail.zip" 
[4] "products.json"      "sample_data"        "transactions.csv"

In [49]:
transactions_imported <- read_csv("transactions.csv")

Rows: 392692 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): StockCode
dbl  (3): InvoiceNo, CustomerID, Quantity
dttm (1): InvoiceDate

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [50]:
head(transactions_imported)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<dbl>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


In [51]:
products_imported <- fromJSON("products.json")

In [52]:
head(products_imported)

,StockCode,Description,UnitPrice
,<chr>,<chr>,<dbl>
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
2,71053,WHITE METAL LANTERN,3.39
3,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
5,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
6,22752,SET 7 BABUSHKA NESTING BOXES,7.65


In [53]:
customers_imported <- read_excel("customers.xlsx")

In [54]:
head(customers_imported)

CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


In [55]:
cat("Transactions:", dim(transactions_imported), "\n")
cat("Products:", dim(products_imported), "\n")
cat("Customers:", dim(customers_imported), "\n")

Transactions: 392692 5 
Products: 3665 3 
Customers: 4338 2 


In [56]:
sales_data <- transactions_imported %>%
  left_join(
    products_imported,
    by = "StockCode"
  )

In [57]:
dim(sales_data)

[1] 392692      7

In [58]:
head(sales_data)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice
<dbl>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65


In [59]:
sum(is.na(sales_data$Description))

[1] 0

In [60]:
sum(is.na(sales_data$UnitPrice))

[1] 0

In [61]:
sales_data <- sales_data %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

In [62]:
head(sales_data)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue
<dbl>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,15.30


In [63]:
sales_data %>%
  select(Quantity, UnitPrice, Revenue) %>%
  head()

Quantity,UnitPrice,Revenue
<dbl>,<dbl>,<dbl>
6,2.55,15.30
6,3.39,20.34
8,2.75,22.00
6,3.39,20.34
6,3.39,20.34
2,7.65,15.30


In [64]:
final_data <- sales_data %>%
  left_join(
    customers_imported,
    by = "CustomerID"
  )

In [65]:
dim(final_data)

[1] 392692      9

In [66]:
head(final_data)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue,Country
<dbl>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>,<chr>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30,United Kingdom
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,20.34,United Kingdom
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,22.00,United Kingdom
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34,United Kingdom
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34,United Kingdom
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,15.30,United Kingdom


In [67]:
sum(is.na(final_data$Country))

[1] 0

In [68]:
final_data %>%
  filter(is.na(Country)) %>%
  select(CustomerID) %>%
  distinct() %>%
  head()

CustomerID
<dbl>


Task 3: Sales and Customer Analysis

3.1 Total Sales Revenue

The total revenue generated from the cleaned dataset is approximately:

9,546,219


3.2 Top 5 Products by Revenue
Rank	Product	Revenue

1	PAPER CRAFT, LITTLE BIRDIE	168,469.60

2	REGENCY CAKESTAND 3 TIER	135,495.30

3	WHITE HANGING HEART T-LIGHT HOLDER	93,745.65

4	MEDIUM CERAMIC TOP STORAGE JAR	81,032.64

5	JUMBO BAG RED RETROSPOT	76,028.70


Observation: PAPER CRAFT, LITTLE BIRDIE generated the highest revenue among the products.

In [69]:
total_revenue <- final_data %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE)
  )

total_revenue

Total_Revenue
<dbl>
9546219


In [70]:
top_products <- final_data %>%
  group_by(StockCode, Description) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

In [71]:
top_products

StockCode,Description,Total_Revenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
22423,REGENCY CAKESTAND 3 TIER,135495.30
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93745.65
23166,MEDIUM CERAMIC TOP STORAGE JAR,81032.64
85099B,JUMBO BAG RED RETROSPOT,76028.70


In [72]:
top_countries <- final_data %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

3.3 Top 5 Countries by Revenue
Rank	Country	Revenue

1	United Kingdom	7,868,408.70

2	Netherlands	329,130.80

3	EIRE	287,260.90

4	Germany	233,804.10

5	France	203,628.30

Observation: The United Kingdom is the dominant market in terms of revenue.

In [73]:
top_countries

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,7868408.7
Netherlands,329130.8
EIRE,287260.9
Germany,233804.1
France,203628.3


In [74]:
top_customers <- final_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Purchase_Value)) %>%
  slice_head(n = 5)

3.4 Top 5 Customers by Purchase Value
Rank	CustomerID	Purchase Value

1	18102	383,153.00

2	14646	323,767.50

3	17450	171,051.90

4	16446	168,472.50

5	14911	155,092.00


Observation: Customer 18102 generated the highest total purchase value.

In [75]:
top_customers

CustomerID,Total_Purchase_Value
<dbl>,<dbl>
18102,383153.0
14646,323767.5
17450,171051.9
16446,168472.5
14911,155092.0


In [76]:
customer_value <- final_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  )

In [77]:
nrow(customer_value)

[1] 4338

In [78]:
quantile(
  customer_value$Total_Purchase_Value,
  probs = c(0.25, 0.50, 0.75),
  na.rm = TRUE
)

25%      50%      75% 
 317.835  703.570 1744.907

In [79]:
q1 <- 317.835
q2 <- 703.57
q3 <- 1744.9075

customer_value <- customer_value %>%
  mutate(
    Customer_Category = case_when(
      Total_Purchase_Value <= q1 ~ "Low Value",
      Total_Purchase_Value <= q2 ~ "Medium Value",
      Total_Purchase_Value <= q3 ~ "High Value",
      TRUE ~ "Premium"
    )
  )

In [80]:
customer_value %>%
  count(Customer_Category)

Customer_Category,n
<chr>,<int>
High Value,1084
Low Value,1085
Medium Value,1084
Premium,1085


Customer Value Classification

Customers were classified using quartile-based thresholds calculated from total purchase values.

Category	Threshold

Low Value	≤ 317.835

Medium Value	317.835 – 703.57

High Value	703.57 – 1,744.9075

Premium	> 1,744.9075


The classification was implemented using the case_when() function.

Customer Distribution

Category	Customers

Low Value	1,085

Medium Value	1,084

High Value	1,084

Premium	1,085

Total	4,338

In [81]:
final_data <- final_data %>%
  left_join(
    customer_value %>%
      select(
        CustomerID,
        Customer_Category
      ),
    by = "CustomerID"
  )

In [82]:
head(final_data)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Revenue,Country,Customer_Category
<dbl>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>,<chr>,<chr>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30,United Kingdom,Premium
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,20.34,United Kingdom,Premium
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,22.00,United Kingdom,Premium
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34,United Kingdom,Premium
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34,United Kingdom,Premium
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,15.30,United Kingdom,Premium


In [83]:
dim(final_data)

[1] 392692     10

In [84]:
market_performance <- final_data %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue))

In [85]:
head(market_performance, 5)

Country,Total_Revenue
<chr>,<dbl>
United Kingdom,7868408.7
Netherlands,329130.8
EIRE,287260.9
Germany,233804.1
France,203628.3


Market Performance
High-Performing Market

United Kingdom

Revenue: 7,868,408.70

The United Kingdom is the strongest market and contributes substantially more revenue than the other countries.

Underperforming Market

Saudi Arabia

Revenue: 145.92

Saudi Arabia generated the lowest revenue among the countries in the dataset. The company could investigate demand, marketing reach, product availability, and distribution opportunities in this market.

In [86]:
tail(market_performance, 5)

Country,Total_Revenue
<chr>,<dbl>
Brazil,1184.43
RSA,971.82
Czech Republic,875.46
Bahrain,544.80
Saudi Arabia,145.92


Task 4: SQLite Database

The cleaned and integrated dataset was stored in a SQLite database:

Database: retail_sales.db

Table: retail_sales

The table contains:

392,692 records

10 columns

Database Columns

InvoiceNo

StockCode

CustomerID

Quantity

InvoiceDate

Description

UnitPrice

Revenue

Country

Customer_Category

SQLite Verification

The final dataset was successfully stored in the SQLite database retail_sales.db under the table retail_sales.

Database file exists: TRUE

Table name: retail_sales

Total records: 392,692

Total columns: 10

SQLite connection closed successfully after querying.

In [87]:
con <- dbConnect(
  SQLite(),
  "retail_sales.db"
)

In [88]:
dbIsValid(con)

[1] TRUE

In [89]:
dbWriteTable(
  con,
  "retail_sales",
  final_data,
  overwrite = TRUE
)

In [90]:
dbListTables(con)

[1] "retail_sales"

In [91]:
dbListFields(con, "retail_sales")

[1] "InvoiceNo"         "StockCode"         "CustomerID"       
 [4] "Quantity"          "InvoiceDate"       "Description"      
 [7] "UnitPrice"         "Revenue"           "Country"          
[10] "Customer_Category"

In [92]:
dbGetQuery(
  con,
  "SELECT COUNT(*) AS Total_Rows FROM retail_sales"
)

Total_Rows
<int>
392692


In [93]:
query1 <- "
SELECT
    CustomerID,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY CustomerID
ORDER BY Total_Revenue DESC
LIMIT 5
"

SQL Query 1: Top 5 Customers by Revenue

The SQL query was executed from R using DBI and RSQLite.

In [94]:
top5_customers_sql <- dbGetQuery(con, query1)

top5_customers_sql

CustomerID,Total_Revenue
<dbl>,<dbl>
18102,383153.0
14646,323767.5
17450,171051.9
16446,168472.5
14911,155092.0


SQL Query 2: Revenue by Country

In [95]:
query2 <- "
SELECT
    Country,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY Country
ORDER BY Total_Revenue DESC
"

revenue_by_country_sql <- dbGetQuery(con, query2)

head(revenue_by_country_sql, 10)

,Country,Total_Revenue
,<chr>,<dbl>
1,United Kingdom,7868408.65
2,Netherlands,329130.77
3,EIRE,287260.87
4,Germany,233804.06
5,France,203628.28
6,Australia,157316.33
7,Spain,62080.85
8,Switzerland,57268.14
9,Belgium,43098.99


In [96]:
dbDisconnect(con)

In [97]:
dbIsValid(con)

[1] FALSE

In [98]:
file.exists("retail_sales.db")

[1] TRUE

In [99]:
file.info("retail_sales.db")$size

[1] 37822464

Three Important Business Insights
1. United Kingdom is the dominant market

The United Kingdom generated approximately 7.87 million in revenue, making it the strongest market by a significant margin.

2. High-value customers and products contribute significantly to sales

Customer 18102 generated 383,153.00, while PAPER CRAFT, LITTLE BIRDIE generated 168,469.60. Identifying and retaining high-value customers and high-performing products can support revenue growth.

3. Market performance varies significantly

The United Kingdom generated approximately 7.87 million, whereas Saudi Arabia generated only 145.92. This large difference indicates substantial variation in market performance and highlights opportunities to investigate and improve weaker markets.

Conclusion

The heterogeneous retail datasets were successfully imported, cleaned, integrated, analyzed, and stored using R and SQLite. The final dataset contained 392,692 valid transaction records and 10 attributes. dplyr was used for data cleaning, joining, aggregation, and customer segmentation, while SQLite was used for persistent storage and SQL-based analysis. The analysis identified the company's top products, customers, and markets and provided business insights that can support retail decision-making.